# Практическое задание 2. Мошенничество по банковским картам

**Дисциплина «ИИ и машинное обучение»** · НИУ ВШЭ, Высшая школа бизнеса

Магистратура «Бизнес-аналитика и цифровые решения в финансах»

---

Служба безопасности банка каждый месяц вручную разбирает 2 % операций по
картам — те, что модель считает самыми подозрительными. Нужно оценить,
насколько подозрительна каждая операция декабря 2019 года, так чтобы разбор
нашёл как можно больше мошенничеств. Отметки о мошенничестве есть
в истории; отметку любой операции ноября можно получить, заказав её
проверку у бота курса, но проверки стоят денег.

У каждого свой вариант: операции у всех одни, а отметки о мошенничестве —
свои. Номер варианта смотрите в файле
`распределение_вариантов_по_студентам.xlsx` в нашем чате.

Что требуется, как считается итог и как ставится оценка — в тексте
задания: https://github.com/Rimest/ml_ai_course_gsb_hse/blob/main/homeworks/hw2_fraud/README.md

**Что сдать:** этот ноутбук `ФАМИЛИЯ_ИО.ipynb` с выполненными ячейками и файл
`ФАМИЛИЯ_ИО_predictions.csv` с оценками.

## Преамбула

Впишите фамилию с инициалами и номер варианта и выполните ячейку. Сид `SEED`
подставляйте везде, где есть параметр `random_state`: ноутбук должен при
повторном запуске сверху вниз давать те же числа.

In [ ]:
## ── воспроизводимость ──────────────────────────────────────────────
import random

import numpy as np

STUDENT_NAME = 'Галяминская_АВ'  # например, "ИВАНОВ_ИИ"
SEED = 33
VARIANT = 33
random.seed(SEED)      # стандартный модуль random
np.random.seed(SEED)   # numpy

# Своего генератора у pandas и scikit-learn нет — оба берут случайность
# из numpy, так что строка выше покрывает и их. Но там, где параметр
# random_state есть, передавайте SEED явно.

## Данные

Данные лежат на Яндекс Диске: https://disk.yandex.ru/d/tetGliBdgbgjUA

Скачайте оттуда четыре файла и положите рядом с ноутбуком (в Colab —
загрузите в папку сессии). Ячейка ниже читает их в четыре таблицы.

| Таблица | Файл | Что внутри |
|---|---|---|
| `history` | `transactions_history.parquet` | операции января 2017 — октября 2019 |
| `pool` | `transactions_pool.parquet` | операции ноября 2019, отметки — только через проверки |
| `test` | `transactions_test.parquet` | операции декабря 2019, по ним сдаются оценки |
| `labels` | `history_labels_NN.parquet` из папки `labels` | отметки истории вашего варианта: `операция`, `мошенничество` |

`NN` — номер вашего варианта двумя цифрами: `history_labels_07.parquet`,
например. Описание каждого столбца — в тексте задания.

In [ ]:
## ── данные: файлы с Яндекс Диска, положенные рядом с ноутбуком ─────
import pandas as pd

history = pd.read_parquet("transactions_history.parquet")
pool = pd.read_parquet("transactions_pool.parquet")
test = pd.read_parquet("transactions_test.parquet")
labels = pd.read_parquet(f"history_labels_{VARIANT:02d}.parquet")

print(f"история: {len(history):_} операций, отметок: {len(labels):_}".replace("_", " "))
print(f"ноябрь:  {len(pool):_} операций".replace("_", " "))
print(f"декабрь: {len(test):_} операций".replace("_", " "))

## Решение

In [ ]:
count = history['операция'].isin(labels['операция']).sum()
print(f"Количество строк с совпадениями: {count}, всего: {history['операция'].count()}")

In [ ]:
history.info(show_counts=True)

In [ ]:
history['индекс'] = history['индекс'].fillna(0)
history['штат'] = history['штат'].fillna('Not Stated')
history['ошибки'] = history['ошибки'].fillna(0)
history['year'] = history['время'].dt.year
history['month'] = history['время'].dt.month
history['hour'] = history['время'].dt.hour
history['dayofweek'] = history['время'].dt.dayofweek
history = history.drop(columns=['время'])

CatBoost

In [ ]:
nov = history.merge(labels, how = 'inner', on = 'операция')

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

X = nov.drop(columns=['мошенничество', 'операция'])
Y = nov['мошенничество']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=33, stratify=Y )
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Категориальные признаки: {cat_features}")

train_pool = Pool(X_train, Y_train, cat_features=cat_features)
test_pool = Pool(X_test, Y_test, cat_features=cat_features)
model = CatBoostClassifier(
    iterations=500,          
    learning_rate=0.1,       
    depth=10,                 
    loss_function='Logloss', 
    eval_metric='AUC',       
    random_seed=SEED,
    verbose=100 ,          
    early_stopping_rounds=50,
    auto_class_weights='Balanced'
)
print("Начинаем обучение...")
model.fit(train_pool, eval_set=test_pool, use_best_model=True)
Y_pred = model.predict(X_test)            
Y_pred_proba = model.predict_proba(X_test)[:, 1] 

acc = accuracy_score(Y_test, Y_pred)
roc_auc = roc_auc_score(Y_test, Y_pred_proba)
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC:  {roc_auc:.4f}")


Другая проба

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test) 
print(classification_report(Y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(Y_test, y_pred_proba)}")
a = confusion_matrix(Y_test, y_pred)
print("Confusion Matrix:")
print(a)


Блок для отправки результатов на проверку(ПРИ ПОВТОРНОМ ЗАПУСКЕ ВЫДАСТ ОШИБКУ, НУЖНО ЗАМЕНИТЬ N)

In [ ]:
#ЭТОТ БЛОК КОДА ЗАККОМЕНТИРОВАН ПОТОМУ ЧТО СЛУЖИЛ ДЛЯ ОТПРАВКИ ПРОВЕРОК, ПРИ НЕИЗМЕННОМ N И ПОВТОРНОМ ЗАПУСКЕ
# ВЫДАСТ ОШИБКУ. ПРИ НЕОБХОДИМОСТИ ЗАПУСКА ЗАМЕНИТЬ N НА ТРЕТЬЕЙ СНИЗУ СТРОКЕ НА ЦИФРУ 4
"""
import pandas as pd
import numpy as np
import glob

def cost(n):
    "Стоимость n проверок за всё задание, $."
    return 50 * n * (1 + n / 1000)

def dollars(x):
    "12345.6 -> «12 346 $»."
    return f"{x:,.0f} $".replace(",", " ")

def check_answers():
    "Все ответы бота — файлы check_*.csv рядом с ноутбуком — одной таблицей."
    files = sorted(glob.glob("check_*.csv"))
    if not files:
        return pd.DataFrame({"операция": pd.Series(dtype="int64"), "мошенничество": pd.Series(dtype="int64")})
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def purchase_file(ids, n):
    "Записать номера операций ноября в purchase_n.csv; отправьте его боту после команды /check."
    ids = pd.Series(np.asarray(ids)).astype("int64")
    done = check_answers()["операция"]
    problems = []
    if ids.duplicated().any():
        problems.append(f"номера повторяются: {ids.duplicated().sum()}")
    if not ids.isin(pool["операция"]).all():
        problems.append(f"не операции ноября: {(~ids.isin(pool['операция'])).sum()}")
    if ids.isin(done).any():
        problems.append(f"уже проверены: {ids.isin(done).sum()}")
    if len(ids) > 1000:
        problems.append(f"в одной покупке не больше 1 000 операций, а здесь {len(ids)}")
    if problems:
        raise ValueError("; ".join(problems))
    path = f"purchase_{n}.csv"
    pd.DataFrame({"операция": ids}).to_csv(path, index=False, encoding="utf-8")
    total = len(done) + len(ids)
    print(f"{path}: {len(ids)} операций. После покупки проверок станет {total}, "
          f"их стоимость {dollars(cost(total))} (эта покупка: {dollars(cost(total) - cost(len(done)))})")
    return path

print(f"Ответов на проверки рядом с ноутбуком: {len(check_answers())}")
pool = pd.read_parquet("transactions_pool.parquet")
print(f"Загружено операций за ноябрь: {len(pool)}")

pool_prep = pool.copy()
pool_prep['индекс'] = pool_prep['индекс'].fillna(0)
pool_prep['штат'] = pool_prep['штат'].fillna('Not Stated')
pool_prep['ошибки'] = pool_prep['ошибки'].fillna(0)
pool_prep['year'] = pool_prep['время'].dt.year
pool_prep['month'] = pool_prep['время'].dt.month
pool_prep['hour'] = pool_prep['время'].dt.hour
pool_prep['dayofweek'] = pool_prep['время'].dt.dayofweek
pool_prep = pool_prep.drop(columns=['время'])

if 'операция' in pool_prep.columns:
    # Сохраняем ID отдельно, они  нужны для отправки в  бот
    pool_ids = pool_prep['операция'].copy()
    pool_prep = pool_prep.drop(columns=['операция'])

X_pool = pool_prep.copy() 

pool_probas = model.predict_proba(X_pool)[:, 1]
results = pd.DataFrame({
    'операция': pool_ids, 
    'proba': pool_probas
})

low_bound = 0.35
high_bound = 0.65

suspicious = results[(results['proba'] >= low_bound) & (results['proba'] <= high_bound)].copy()

suspicious['dist'] = (suspicious['proba'] - 0.5).abs()
suspicious = suspicious.sort_values(by='dist')

print(f"Всего операций в зоне неопределенности [{low_bound}, {high_bound}]: {len(suspicious)}")
already_checked = check_answers()['операция']
suspicious = suspicious[~suspicious['операция'].isin(already_checked)]
ids_to_check = suspicious['операция'].head(1000).values

print(f"Отправляем на проверку: {len(ids_to_check)} операций")

if len(ids_to_check) > 0:
    purchase_file(ids_to_check, n=2)
else:
    print("Нет операций для проверки (возможно, все уже проверены или зона пуста).")
"""

In [ ]:
# то же самое что и с блоком выше
"""answers = pd.read_csv('check_1.csv')
# Объединяем ответы с нашими предсказаниями (по ID операции)
merged = answers.merge(results, on='операция', how='left')

# Смотрим, какие вероятности были у реальных мошенников
frauds_probas = merged[merged['мошенничество'] == 1]['proba']
print(f"Вероятности у настоящих мошенников (из 1000 проверенных):\n{frauds_probas.sort_values()}")
"""

In [ ]:
"""
# Смотрим распределение вероятностей для 1000 проверенных операций
import matplotlib.pyplot as plt
merged = check_answers().merge(results, on='операция', how='left')
frauds = merged[merged['мошенничество'] == 1]['proba']
normals = merged[merged['мошенничество'] == 0]['proba']

print(f"Мошенники (21 шт): \n{frauds.describe()}\n")
print(f"Нормальные (979 шт): \n{normals.describe()}\n")
plt.hist(normals, bins=50, alpha=0.5, label='Нормальные', color='blue')
plt.hist(frauds, bins=50, alpha=0.5, label='Мошенники', color='red')
plt.legend()
plt.title('Распределение вероятностей в серой зоне')
plt.show()
"""

In [ ]:
"""sample_ids = merged['операция'].head(5)

# Смотрим на их исходные данные в pool
print(pool[pool['операция'].isin(sample_ids)])
"""

In [ ]:
#Во второй раз корректирую порог
"""
high_risk = results[results['proba'] >= 0.6].copy()
high_risk = high_risk.sort_values(by='proba', ascending=False)
already_checked = check_answers()['операция']
high_risk = high_risk[~high_risk['операция'].isin(already_checked)]
ids_to_check_2 = high_risk['операция'].head(1000).values
print(f"Найдено операций с вероятностью > 0.6: {len(high_risk)}")
print(f"Отправляем на проверку: {len(ids_to_check_2)}")
if len(ids_to_check_2) > 0:
    purchase_file(ids_to_check_2, n=2)

"""

In [ ]:
import matplotlib.pyplot as plt


#Выведу график распределения вероятностей мошенников и нормальных по всем проверкам (check_1 и check_2)

"""
all_answers = check_answers()
merged_all = all_answers.merge(results, on='операция', how='left')
frauds = merged_all[merged_all['мошенничество'] == 1]['proba']
normals = merged_all[merged_all['мошенничество'] == 0]['proba']
print(f"Всего проверено мошенников: {len(frauds)}")
print(f"Всего проверено нормальных: {len(normals)}")
plt.figure(figsize=(10, 5))
plt.hist(normals, bins=50, alpha=0.5, label='Нормальные', color='blue', density=True)
plt.hist(frauds, bins=50, alpha=0.5, label='Мошенники', color='red', density=True)
plt.title('Распределение вероятностей: Мошенники vs Нормальные')
plt.xlabel('Вероятность (proba)')
plt.ylabel('Плотность')
plt.legend()
plt.show()
"""

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

#Задаем новый порог
NEW_THRESHOLD = 0.7
y_pred_new = (Y_pred_proba >= NEW_THRESHOLD).astype(int)
print(classification_report(Y_test, y_pred_new))
cm = confusion_matrix(Y_test, y_pred_new)
print("Confusion Matrix:")
print(cm)
print(f"\nROC-AUC (не зависит от порога): {roc_auc_score(Y_test, Y_pred_proba):.4f}")

#Попробуем посчитать условную экономическую стоимость
tn, fp, fn, tp = confusion_matrix(Y_test, y_pred_new).ravel()
COST_FN = 10000  # Убыток, если пропустили мошенника (потерянная сумма, берется в условных единицах)
COST_FP = 500   # Убыток, если ошибочно заблокировали нормального (издержки поддержки, берется в условных единицах)  
total_loss = (fn * COST_FN) + (fp * COST_FP)
print(f"Условный общий убыток: {total_loss:,} условных единиц")

Лучшим порогом, исходя из расчета экономики, принят 0.7

In [ ]:
import pandas as pd
import numpy as np

test_prep = test.copy()
test_prep['индекс'] = test_prep['индекс'].fillna(0)
test_prep['штат'] = test_prep['штат'].fillna('Not Stated')
test_prep['ошибки'] = test_prep['ошибки'].fillna(0)
test_prep['year'] = test_prep['время'].dt.year
test_prep['month'] = test_prep['время'].dt.month
test_prep['hour'] = test_prep['время'].dt.hour
test_prep['dayofweek'] = test_prep['время'].dt.dayofweek
test_prep = test_prep.drop(columns=['время'])

test_ids = test_prep['операция'].copy()
test_prep = test_prep.drop(columns=['операция'])
X_test_dec = test_prep 
score_test = model.predict_proba(X_test_dec)[:, 1]


## Покупка проверок

Проверки заказываются у бота: команда `/check`, затем файл с номерами
операций ноября, затем `/yes`. Бот присылает файл `check_N.csv` с
отметками — положите его рядом с ноутбуком (в Colab — загрузите в папку
сессии).

Ячейка ниже даёт три функции:

| Функция | Что делает |
|---|---|
| `purchase_file(ids, n)` | записывает номера операций ноября в `purchase_n.csv` (`n` — номер покупки: 1, 2, 3…) и проверяет формат: номера из ноября, без повторов, без уже проверенных, не больше 1 000 |
| `check_answers()` | читает все файлы `check_*.csv` рядом с ноутбуком в одну таблицу `операция`, `мошенничество` |
| `cost(n)` | стоимость n проверок за всё задание, $ |

In [ ]:
## ── покупка проверок: файл для бота и ответы бота ──────────────────
import glob


def cost(n):
    "Стоимость n проверок за всё задание, $."
    return 50 * n * (1 + n / 1000)


def dollars(x):
    "12345.6 -> «12 346 $»."
    return f"{x:,.0f} $".replace(",", " ")


def check_answers():
    "Все ответы бота — файлы check_*.csv рядом с ноутбуком — одной таблицей."
    files = sorted(glob.glob("check_*.csv"))
    if not files:
        return pd.DataFrame({"операция": pd.Series(dtype="int64"), "мошенничество": pd.Series(dtype="int64")})
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def purchase_file(ids, n):
    "Записать номера операций ноября в purchase_n.csv; отправьте его боту после команды /check."
    ids = pd.Series(np.asarray(ids)).astype("int64")
    done = check_answers()["операция"]
    problems = []
    if ids.duplicated().any():
        problems.append(f"номера повторяются: {ids.duplicated().sum()}")
    if not ids.isin(pool["операция"]).all():
        problems.append(f"не операции ноября: {(~ids.isin(pool['операция'])).sum()}")
    if ids.isin(done).any():
        problems.append(f"уже проверены: {ids.isin(done).sum()}")
    if len(ids) > 1000:
        problems.append(f"в одной покупке не больше 1 000 операций, а здесь {len(ids)}")
    if problems:
        raise ValueError("; ".join(problems))
    path = f"purchase_{n}.csv"
    pd.DataFrame({"операция": ids}).to_csv(path, index=False, encoding="utf-8")
    total = len(done) + len(ids)
    print(f"{path}: {len(ids)} операций. После покупки проверок станет {total}, "
          f"их стоимость {dollars(cost(total))} (эта покупка: {dollars(cost(total) - cost(len(done)))})")
    return path


print(f"Ответов на проверки рядом с ноутбуком: {len(check_answers())}")

## Файл сдачи

Ячейка ниже собирает `ФАМИЛИЯ_ИО_predictions.csv`. Формат одинаков у всех,
поэтому код готовый — выполните ячейку, менять её не требуется. Кроме полей
преамбулы к этому моменту должны быть две величины:

| Имя | Что это |
|---|---|
| `test` | таблица операций декабря, со столбцом `операция` |
| `score_test` | оценки подозрительности операций `test` в том же порядке: чем больше, тем подозрительнее |

Получится файл с тремя столбцами:

```
вариант,операция,оценка
7,5022410,0.003512
7,5022411,0.418877
```

In [ ]:
## ── файл сдачи ─────────────────────────────────────────────────────
predictions = pd.DataFrame({
    "вариант": VARIANT,
    "операция": test["операция"].values,
    "оценка": np.asarray(score_test, dtype=float),
})
predictions.to_csv(f"{STUDENT_NAME}_predictions.csv", index=False, encoding="utf-8")

## ── проверка перед сдачей ──────────────────────────────────────────
errors = []
if len(predictions) != 146579:
    errors.append(f"в файле {len(predictions)} строк, а в декабре 146 579 операций")
if predictions["операция"].duplicated().any():
    errors.append("одна и та же операция встречается несколько раз")
if not np.isfinite(predictions["оценка"]).all():
    errors.append("оценки должны быть числами без пропусков")
if not isinstance(VARIANT, int) or not 1 <= VARIANT <= 57:
    errors.append("номер варианта должен быть числом от 1 до 57")

n_checks = len(check_answers())
print(f"Строк в файле: {len(predictions)}")
print(f"Проверок куплено (по файлам рядом с ноутбуком): {n_checks}, их стоимость {dollars(cost(n_checks))}")
print("Проверка: " + ("всё в порядке" if not errors else "; ".join(errors)))
predictions.head()

## Как оценивается работа

За работу — до 10 баллов.

| За что | Баллов | Как считается |
|---|---|---|
| Итог | 8 | 8 × (доля разрыва / 0,80) ^ 1,5, с точностью до 0,1; не больше 8 и не меньше 0 |
| Формат сдачи | 1 | `ФАМИЛИЯ_ИО_predictions.csv` прошёл формальную проверку целиком |
| Ноутбук | 1 | приложен |

```
итог         = 500 $ × пойманные мошенничества − стоимость(проверок)
доля разрыва = (итог − итог бейзлайна) / (итог потолка − итог бейзлайна)
```

Пойманные — число мошенничеств среди 2 931 операции декабря
с наибольшей оценкой. Бейзлайн и потолок описаны в тексте задания. Если в
коде ноутбука есть модели не из лекций 5–6, за итог ставится 0.

## Обоснование

Ответьте на вопросы:

1. **Что вы сделали с данными и почему.** Что добавили, преобразовали или
   отбросили и как проверяли, что это помогает.
2. **Какие модели сравнивали.** Какие модели из лекций 5–6 пробовали, как
   проверяли их качество, какие параметры подбирали и что это дало.
   Почему выбрали итоговую модель.
3. **Какие проверки заказали.** Сколько, каких операций и почему именно
   столько.
4. **Какой итог вы ждёте.** Сколько мошенничеств поймает разбор декабря,
   сколько стоили проверки и почему результат может оказаться хуже ожидаемого.
5. **Как вы использовали ИИ.** Какими ИИ-инструментами пользовались
   (чат-ассистенты, подсказки кода в редакторе и другие) и на каких шагах
   работы. Для каждого шага — что именно вы просили, что взяли из ответа,
   что проверили или исправили сами и почему. Если ИИ не использовали,
   так и напишите.

1. Почистила данные и далее отталкивалась от тестируемой модели. Для random-forest отбросила как операции, так и индексы(потому что много пропусков и столбец в целом малоинформативен, остальное при переполнении количества в 10000 nunique отбрасывала за низкой информативностью, либо же преобразовывала с целью разделения по регионам) Так как CatBoost принимает и не числовые признаки, для него только дозаполнила пропуски.
2. Пробовала разные модели: от обычного random-forest до CatBoost и сравнила показатели. Лучший результат показал CatBoost - для него также варьировала по количеству деревьев, их глубине, размеру останова, балансировку классов. Все это в совокупности дало увелицичения ROC-AUC с 0.86(показатель близкий к результату random-forest) до 0.873. 
3. Заказала 2 проверки. Сначала заказала проверку операций, находящихся в диапазоне вероятности риска до 0.35 до 0.65. По результатам проверки выяснилось, что CatBoost в основном определяет их как белый шум, и действительно мошенников там мало. Вторая проверка была создана для проверки адекватности модели на вероятности, близкой к 1. Результаты показали хорошие показатели, поэтому далее только подбирала порог. Общее количество проверенных операций - 2 000, их стоимость - 300 000.
4. Ожидаю, что поймает не менее 65%. Порог подбирался, исходя из финансового влияния.
5. ИИ не использовала, но использовала личные шаблоны для быстрого построения моделей, подсказки кода в редакторе помогали мне просто быстро заменить одну переменную на другую.